# South Movie → Telegram | New channel only

This notebook is for the **South Movie** repository and the newly created Telegram channel **[@southdhamal](https://t.me/southdhamal)**. It does **not** contain, import, or upload your Bollywood Movie migration.

- Source: `https://github.com/kawakaze430/south-movie` (fixed commit `a4dbc229fa5cc2563801bfc7ba7d416fb8b96138`).
- Target: **@southdhamal**, exact title **South Movie**. The bot must be an administrator with **Post Messages** permission. You can keep the same bot in Bollywood Movie.
- Kaggle: **Internet ON**, **TELEGRAM_BOT_TOKEN** in Add-ons → Secrets with the checkbox enabled. **No dataset upload, channel ID, GitHub token, or Google Drive needed.**
- To run on Kaggle's server after closing your browser: **Save Version → Save & Run All**. Using **Run All** in a Draft Session does not provide the same background execution guarantee.

**Before starting:** ensure you have the right to republish these posts and that South Movie does not already contain posts from this exact migration. The marker `SOUTH_MOVIE_MIGRATION_SETUP_2026` is fine to leave in the new channel.

**What it posts:** ordinary exported messages with substantive text or supported photo/document media. **When Telegram did not export a referenced media file**, the notebook sends that record as text-only if it has a meaningful cleaned caption; if it has no caption, it skips it and records why. It never invents a photo or uses a thumbnail to replace a missing original. It removes Telegram channel promotions and `t.me` links while preserving other URLs, titles, and other text from the export. It uses original photos, never thumbnails. Ordinary service/action entries and channel-promotion-only messages are excluded and reported. Very long captions are split into an attached follow-up message to avoid dropping links.

**Media completeness report:** `south_movie_missing_media_audit.json` records the source message IDs whose media was not exported or is absent from the pinned GitHub snapshot, and whether a text-only fallback was sent or the post skipped. To obtain those missing original photos/videos, re-export them from Telegram with media included; no script can reconstruct unavailable files.

**Checkpoint / duplicate prevention:** a JSON checkpoint is written after every confirmed Telegram API send to `/kaggle/working/progress_south_movie_southdhamal.json`. It includes per-message Telegram message IDs, so a long interrupted run can be resumed. **Do NOT launch a second run or rerun an older version without attaching its latest checkpoint as Kaggle Input.** If the result of a send is uncertain, the notebook stops and asks for manual reconciliation, rather than risk duplicate posts.

*The notebook has not performed an authenticated Telegram upload. Preparing files, plan checks, and dry-run tests do not send messages.*
**After the run:** check the plan summary and media audit in Kaggle Outputs. The total posted is not necessarily equal to source record count because Telegram export service entries, promotion-only entries, and captionless unavailable-media entries are not sent.


In [ ]:
#@title 1 — Configuration and download the pinned South Movie export from GitHub
from pathlib import Path
import os, json, re, io, sys, time, shutil, hashlib, zipfile, mimetypes, urllib.parse, tempfile
import requests

REPO = 'kawakaze430/south-movie'
REF = 'a4dbc229fa5cc2563801bfc7ba7d416fb8b96138'  # Fixed snapshot of main from 2026-11-16
EXPECTED_JSON_GIT_SHA = '9d007411d3dfa271251c301633e82e279c96e0f9'
TARGET_USERNAME = '@southdhamal'
EXPECTED_TARGET_TITLE = 'South Movie'
OLD_BOLLYWOOD_CHANNEL_ID = -1004492918137
RUN_MODE = 'SEND_ALL'  # For a no-send inspection change to 'PREVIEW'.
PAUSE_SECONDS = 3.0

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data/south_movie_local')
TEMP = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else WORK
WORK.mkdir(parents=True, exist_ok=True)
TEMP.mkdir(parents=True, exist_ok=True)
CHECKPOINT_NAME = 'progress_south_movie_southdhamal.json'
CHECKPOINT = WORK / CHECKPOINT_NAME
ARCHIVE = TEMP / ('south_movie_' + REF[:12] + '.zip')
print('South Movie source:', REPO, '@', REF[:12])
print('Target (fixed public username):', TARGET_USERNAME)
print('Mode:', RUN_MODE, '| previous Bollywood Movie progress will NOT be loaded')

def get_secret(label):
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(label)
        if v:
            return str(v).strip()
    except Exception:
        pass
    return os.environ.get(label,'').strip()

def git_blob_sha(data):
    return hashlib.sha1(b'blob ' + str(len(data)).encode() + b'\0' + data).hexdigest()

def download_archive():
    if ARCHIVE.exists() and zipfile.is_zipfile(ARCHIVE):
        print('Reusing cached archive:', ARCHIVE)
        return ARCHIVE
    url = f'https://codeload.github.com/{REPO}/zip/{REF}'
    print('Downloading the South Movie GitHub snapshot (hundreds of MB). No Telegram messages sent yet.')
    for attempt in range(1,4):
        tmp = ARCHIVE.with_suffix('.downloading')
        try:
            with requests.get(url, stream=True, timeout=(25,180), headers={'User-Agent':'SouthMovie-Kaggle-Migration/1.0'}) as r:
                r.raise_for_status()
                total = 0; last_mib = -1
                with open(tmp,'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if not chunk: continue
                        f.write(chunk); total += len(chunk)
                        if total//(32*1024*1024) != last_mib:
                            last_mib = total//(32*1024*1024)
                            if total >= 32*1024*1024:
                                print('Downloaded',total//(1024*1024),'MiB...',flush=True)
            if not zipfile.is_zipfile(tmp): raise ValueError('Downloaded content is not a valid ZIP archive.')
            tmp.replace(ARCHIVE)
            return ARCHIVE
        except Exception as exc:
            print('GitHub archive download attempt',attempt,'failed:',type(exc).__name__,str(exc)[:180],flush=True)
            tmp.unlink(missing_ok=True)
            if attempt==3:
                raise RuntimeError('Could not download GitHub export. Enable Kaggle Internet or retry once later. ZERO posts sent.') from exc
            time.sleep(5*attempt)

ZIP_FILE = download_archive()
SOURCE_ZIP = zipfile.ZipFile(ZIP_FILE,'r')
NAMES = set(SOURCE_ZIP.namelist())
JSON_MATCHES = [x for x in NAMES if x.endswith('/result.json') and x.count('/') == 1]
if len(JSON_MATCHES)!=1:
    raise RuntimeError(f'Expected exactly one root result.json, found {len(JSON_MATCHES)}. ZERO posts sent.')
ROOT = JSON_MATCHES[0][:-len('result.json')]
RAW_JSON = SOURCE_ZIP.read(ROOT+'result.json')
actual_sha = git_blob_sha(RAW_JSON)
if actual_sha != EXPECTED_JSON_GIT_SHA:
    raise RuntimeError('Source result.json Git blob checksum mismatch; snapshot changed or wrong source. ZERO posts sent.')
SOURCE_DATA = json.loads(RAW_JSON)
if SOURCE_DATA.get('type') not in {'public_channel','private_channel'}:
    raise RuntimeError('Expected Telegram channel export, received '+str(SOURCE_DATA.get('type'))+'. ZERO posts sent.')
print('Verified source JSON:',SOURCE_DATA.get('name'),'| records:',len(SOURCE_DATA.get('messages',[])))
print('Total files in source archive:',len(NAMES),'| loaded without extracting thousands of photos.')

In [ ]:
#@title 2 — Prepare cleaned posts; audit unavailable media without blocking good posts (no Telegram sends)
# Telegram Desktop may put an explanatory placeholder in m['file'] instead of a real path.
# If original media is unavailable: preserve useful caption+links as a text-only message,
# or skip contentless placeholders. Record every affected source ID for later re-export.
TELEGRAM_URL = re.compile(r'(?i)(?:https?://)?(?:www\.)?(?:t\.me|telegram\.me|telegram\.dog)/[^\s<>]+')
TELEGRAM_HANDLE = re.compile(r'(?<![\w])@[A-Za-z][A-Za-z0-9_]{4,31}(?![\w])')
OTHER_URL = re.compile(r'(?i)https?://[^\s<>]+')
PROMO_LABEL = re.compile(r'(?i)\b(all private channels|all adult channel|join (?:our|my) (?:telegram )?channel|our (?:new )?telegram channel|subscribe to our channel|channel link|new channels)\b')

# No editing needed: send available captions/links when the original media file was not exported.
MISSING_MEDIA_POLICY = 'TEXT_FALLBACK'  # Alternative: 'SKIP' (all missing-media records skipped)
if MISSING_MEDIA_POLICY not in ('TEXT_FALLBACK','SKIP'):
    raise RuntimeError('Invalid missing media policy')


def flatten_text(x):
    if isinstance(x, str): return x
    if isinstance(x, list): return ''.join(flatten_text(v) for v in x)
    if isinstance(x, dict): return flatten_text(x.get('text',''))
    return ''


def clean_caption(original):
    lines=[]
    for line in original.splitlines():
        had_telegram = bool(TELEGRAM_URL.search(line))
        had_promo = bool(PROMO_LABEL.search(line))
        new = TELEGRAM_URL.sub('', line)
        new = TELEGRAM_HANDLE.sub('', new)
        if had_promo and not OTHER_URL.search(new):
            continue
        if had_telegram and not new.strip(' \t:•-🔞\u200b'):
            continue
        lines.append(new.rstrip())
    result='\n'.join(lines).strip()
    result=re.sub(r'\n{3,}','\n\n',result)
    result=re.sub(r'(?m)^\s*[▬━─—]{3,}\s*$', '────────────', result)
    result=re.sub(r'(?:\n────────────\s*)+$','',result).strip()
    if TELEGRAM_URL.search(result): raise AssertionError('Telegram channel link survived cleaning')
    # A hard check: cleaning must preserve all original non-Telegram download URL occurrences.
    ext = lambda t: [u for u in OTHER_URL.findall(t) if not TELEGRAM_URL.fullmatch(u)]
    before,after=ext(original),ext(result)
    if before!=after:
        raise RuntimeError('Caption cleanup lost/changed external links. Before: '+str(before)+' After: '+str(after))
    return result, len(before)


def is_missing_media_notice(value):
    if not isinstance(value,str): return False
    v=value.casefold().strip()
    return bool(re.search(r'(file|media) not included|change data exporting settings|not downloaded|media file was not included',v))


def resolve_media(m):
    """Return (validated ZIP-relative media path, Telegram kind, problem description)."""
    missing=[]
    # Try full photo first; otherwise valid exported document/video as Telegram document.
    for field,kind in (('photo','photo'),('file','document')):
        value=m.get(field)
        if not value or not isinstance(value,str): continue
        if is_missing_media_notice(value):
            missing.append({'reference':value,'reason':'not_included_in_telegram_export','field':field})
            continue
        value=value.replace('\\','/')
        if value.startswith('/') or '..' in value.split('/') or not value.strip():
            raise RuntimeError('Unsafe media path in source post '+str(m.get('id'))+': '+repr(value))
        path=ROOT+value
        if path not in NAMES:
            missing.append({'reference':value,'reason':'file_absent_from_pinned_github_snapshot','field':field})
            continue
        if SOURCE_ZIP.getinfo(path).file_size==0:
            missing.append({'reference':value,'reason':'empty_media_file','field':field})
            continue
        return value,kind,missing
    return None,None,missing


source_messages = [x for x in SOURCE_DATA.get('messages',[]) if x.get('type')=='message']
seen_ids=set(); POSTS=[]; missing_media_audit=[]
removed_ads=0; photo_count=0; other_media_count=0
text_fallback_count=0; skipped_missing_media_count=0
for m in sorted(source_messages,key=lambda x:(x.get('date_unixtime') or '',int(x.get('id',0)))):
    sid=int(m['id'])
    if sid in seen_ids: raise RuntimeError('Duplicate original Telegram message ID: '+str(sid))
    seen_ids.add(sid)
    orig=flatten_text(m.get('text_entities') or m.get('text',''))
    clean, link_count=clean_caption(orig)
    media_path,media_kind,problems=resolve_media(m)
    if problems:
        fallback = (media_path is None and bool(clean) and MISSING_MEDIA_POLICY=='TEXT_FALLBACK')
        skip = (media_path is None and not fallback)
        missing_media_audit.append({'source_id':sid,'media_problems':problems,
                                    'action': 'text_only_fallback' if fallback else ('skipped' if skip else 'other_exported_media_available'),
                                    'has_caption':bool(clean),'external_links_in_caption':link_count})
        if fallback: text_fallback_count += 1
        if skip: skipped_missing_media_count += 1
    if media_path:
        if media_kind=='photo': photo_count += 1
        else: other_media_count += 1
    if not media_path and not clean:
        removed_ads += 1
        continue
    if not media_path and problems and MISSING_MEDIA_POLICY=='SKIP':
        continue
    POSTS.append({'source_id':sid,'date':m.get('date',''),'caption':clean,
                  'media':media_path,'media_kind':media_kind,'external_links':link_count})

if not POSTS: raise RuntimeError('No valid source messages in this GitHub export. ZERO posts sent.')
if any(TELEGRAM_URL.search(p['caption']) for p in POSTS): raise RuntimeError('Telegram link in outgoing caption')
if int(SOURCE_DATA.get('id',0))==OLD_BOLLYWOOD_CHANNEL_ID: raise RuntimeError('Wrong old source/destination selection')

# IMPORTANT: This digest is tied to the exact pinned GitHub snapshot and chosen fallback plan.
# Old progress from an unrelated run must never be loaded silently.
PLAN_DIGEST = hashlib.sha256(json.dumps({'ref':REF,'repo':REPO,'posts':POSTS},ensure_ascii=False,sort_keys=True).encode()).hexdigest()
MANIFEST = {
    'repo':REPO,'ref':REF,'source_name':SOURCE_DATA.get('name'),'source_id':SOURCE_DATA.get('id'),
    'destination':TARGET_USERNAME,'plan_sha256':PLAN_DIGEST,'source_messages':len(source_messages),
    'outgoing_posts':len(POSTS),'photos':photo_count,'other_media':other_media_count,
    'text_only':len(POSTS)-photo_count-other_media_count,
    'missing_media_policy':MISSING_MEDIA_POLICY,'records_with_media_problems':len(missing_media_audit),
    'text_only_due_to_missing_media':text_fallback_count,
    'skipped_due_to_missing_media':skipped_missing_media_count,
    'skipped_empty_or_channel_promotion_including_media_only_placeholders':removed_ads,
    'non_channel_url_occurrences_preserved':sum(p['external_links'] for p in POSTS)
}
audit_path=WORK/'south_movie_missing_media_audit.json'
audit_path.write_text(json.dumps(missing_media_audit,indent=2,ensure_ascii=False),encoding='utf-8')
(WORK/'south_movie_plan_summary.json').write_text(json.dumps(MANIFEST,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(MANIFEST,indent=2,ensure_ascii=False),flush=True)
print('Media issues recorded in:',audit_path, '| entries:',len(missing_media_audit),flush=True)
for issue in missing_media_audit[:10]:
    print('MISSING MEDIA:',issue['source_id'],'|',issue['action'],'|',issue['media_problems'][0]['reason'],flush=True)
print('First 5 outgoing posts:')
for p in POSTS[:5]: print('Source',p['source_id'],'|',p['media_kind'] or 'text','|',p['caption'][:180].replace('\n',' / '))
print('Last outgoing:', POSTS[-1]['source_id'],'|',POSTS[-1]['caption'][:100].replace('\n',' / '))
print('NO Telegram messages sent during plan preparation.',flush=True)


In [ ]:
#@title 3 — Verify @southdhamal and verify bot is a posting administrator (no sends)
def utf16len(text): return len(text.encode('utf-16-le'))//2

class BotAPI:
    def __init__(self,token):
        if not token or ':' not in token:
            raise RuntimeError('Missing/invalid TELEGRAM_BOT_TOKEN Kaggle secret. Tick the checkbox in Add-ons → Secrets.')
        self._token=token
        self.endpoint='https://api.telegram.org/bot'+token+'/'
        self.session=requests.Session()
        self.session.headers['User-Agent']='SouthMovieKaggleMigration/1.0'
    def call(self,method,data=None,files=None,sending=False):
        for attempt in range(1,7):
            # A 429 retry reuses the same multipart request; rewind the uploaded photo.
            if files:
                for item in files.values():
                    if hasattr(item[1],'seek'): item[1].seek(0)
            try:
                r=self.session.post(self.endpoint+method,data=data or {},files=files,timeout=(20,120 if sending else 45))
            except requests.exceptions.RequestException as ex:
                if sending:
                    raise RuntimeError('UNCERTAIN TELEGRAM DELIVERY: network failed during '+method+'. Inspect target and checkpoint before retrying: '+str(ex).replace(self._token,'[REDACTED]')) from ex
                if attempt==6: raise
                time.sleep(3*attempt)
                continue
            try: payload=r.json()
            except Exception:
                raise RuntimeError(('UNCERTAIN DELIVERY' if sending else 'Invalid Telegram response')+' for '+method+'; HTTP '+str(r.status_code))
            if payload.get('ok'): return payload['result']
            if r.status_code==429 or payload.get('error_code')==429:
                wait=int(payload.get('parameters',{}).get('retry_after',20))+2
                print('Telegram flood control:',method,'retry in',wait,'seconds',flush=True)
                if wait>3600: raise RuntimeError('Telegram wait too long; paused safely')
                time.sleep(wait)
                continue
            why=str(payload.get('description','unknown error'))
            if sending and (r.status_code>=500 or payload.get('error_code',0)>=500):
                raise RuntimeError('UNCERTAIN TELEGRAM DELIVERY: server error during '+method+': '+why)
            raise RuntimeError('Telegram '+method+' failed: '+why)
        raise RuntimeError('Telegram rate limit retries exhausted')

BOT_TOKEN = get_secret('TELEGRAM_BOT_TOKEN')
if RUN_MODE != 'PREVIEW' and not BOT_TOKEN:
    raise RuntimeError('Enable the TELEGRAM_BOT_TOKEN secret for this notebook, then rerun. ZERO posts sent.')
if RUN_MODE not in ('SEND_ALL','PREVIEW'):
    raise RuntimeError('Invalid RUN_MODE')

API = BotAPI(BOT_TOKEN) if BOT_TOKEN else None
DEST = None; BOT = None
if API:
    BOT=API.call('getMe')
    DEST=API.call('getChat',{'chat_id':TARGET_USERNAME})
    print('Authenticated bot:',BOT.get('username'))
    print('Resolved destination:',DEST.get('title'),'| id',DEST.get('id'),'| username',DEST.get('username'))
    if DEST.get('type')!='channel' or (DEST.get('username') or '').lower()!='southdhamal':
        raise RuntimeError('Wrong destination. Expected the public channel @southdhamal. ZERO posts sent.')
    if DEST.get('title','').strip() != EXPECTED_TARGET_TITLE:
        raise RuntimeError('Destination title differs from South Movie. ZERO posts sent.')
    if int(DEST['id']) == OLD_BOLLYWOOD_CHANNEL_ID:
        raise RuntimeError('Refusing to send South Movie posts into Bollywood Movie. ZERO posts sent.')
    membership=API.call('getChatMember',{'chat_id':DEST['id'],'user_id':BOT['id']})
    status=membership.get('status','')
    if not (status=='creator' or (status=='administrator' and membership.get('can_post_messages') is True)):
        raise RuntimeError('Bot is not South Movie channel admin with Post Messages permission. ZERO posts sent.')
    print('DESTINATION VERIFIED:',DEST['title'],'(@'+DEST['username']+') | posting rights OK',flush=True)
elif RUN_MODE=='PREVIEW':
    print('PREVIEW ONLY: bot secret not loaded, no Telegram API calls')

In [ ]:
#@title 4 — Upload South Movie content with strict checkpointing (SEND_ALL)
def cut_utf16(text,limit):
    if not text: return []
    parts=[]; remaining=text
    while remaining:
        if utf16len(remaining)<=limit:
            parts.append(remaining); break
        consumed=0; unitcount=0
        for i,ch in enumerate(remaining):
            n=2 if ord(ch)>0xFFFF else 1
            if unitcount+n>limit: break
            unitcount+=n; consumed=i+1
        piece=remaining[:consumed]
        # Prefer breaks after complete lines; otherwise whitespace; never split URLs.
        k=piece.rfind('\n')
        if k<max(40,len(piece)//2): k=piece.rfind(' ')
        if k<max(40,len(piece)//2): k=consumed
        else: k+=1
        if k==0: raise RuntimeError('Cannot split long Telegram text')
        frag=remaining[:k]
        if re.search(r'(?i)https?://[^\s]*$',frag) and k<consumed:
            k=consumed;frag=remaining[:k]
        if re.search(r'(?i)https?://[^\s]*$',frag) and remaining[k:k+1].strip() and re.match(r'^[^\s]',remaining[k:]):
            raise RuntimeError('A single URL is too long for Telegram text limit')
        parts.append(frag)
        remaining=remaining[k:]
    if ''.join(parts)!=text: raise RuntimeError('Text splitting altered content')
    return parts

def units_for(post):
    t=post['caption']; media=post['media_kind']
    if media:
        chunks=cut_utf16(t,1024) if t else []
        if chunks:
            units=[{'kind':media,'text':chunks[0]}]
            rest=t[len(chunks[0]):]
        else:
            units=[{'kind':media,'text':''}]; rest=''
        units.extend({'kind':'text','text':part} for part in cut_utf16(rest,4096))
        return units
    return [{'kind':'text','text':part} for part in cut_utf16(t,4096)]

UNITS=[]
for p in POSTS:
    for part_no,u in enumerate(units_for(p)):
        UNITS.append({'source_id':p['source_id'],'part':part_no, 'kind':u['kind'],'text':u['text'],'media':p['media']})
print('Planned source posts:',len(POSTS),'| Telegram outbound API messages:',len(UNITS))
print('Additional messages caused by long captions:',len(UNITS)-len(POSTS))

if RUN_MODE=='PREVIEW':
    print('PREVIEW MODE - zero messages sent. Change RUN_MODE in configuration to SEND_ALL when ready.')
else:
    import json as _json
    def load_checkpoint():
        candidates=[]
        if CHECKPOINT.exists(): candidates.append(CHECKPOINT)
        inputdir=Path('/kaggle/input')
        if inputdir.exists(): candidates.extend(inputdir.rglob(CHECKPOINT_NAME))
        if candidates:
            candidates=list({str(p):p for p in candidates}.values())
            # Prefer latest checkpoint with the most confirmed sent units.
            states=[]
            for path in candidates:
                st=_json.loads(path.read_text(encoding='utf-8'))
                states.append((len(st.get('sent',{})),path,st))
            states.sort(key=lambda e:e[0],reverse=True)
            print('Restoring checkpoint:',states[0][1],'| completed units:',states[0][0])
            return states[0][2]
        print('NO PREVIOUS CHECKPOINT FOUND. This starts the SOUTH Movie batch from the first post.')
        print('If earlier South Movie posts were sent, STOP; attach their progress checkpoint to avoid duplicates.')
        return {'version':1,'destination':TARGET_USERNAME,'target_id':DEST['id'],'bot_id':BOT['id'],
                'source_repo':REPO,'source_ref':REF,'plan_sha256':PLAN_DIGEST,
                'sent':{},'pending':None}
    STATE=load_checkpoint()
    for name,expected in [('destination',TARGET_USERNAME),('target_id',DEST['id']),('bot_id',BOT['id']),
                          ('source_repo',REPO),('source_ref',REF),('plan_sha256',PLAN_DIGEST)]:
        if STATE.get(name)!=expected:
            raise RuntimeError('Checkpoint mismatch for '+name+'; refusing to send. ZERO new posts sent.')
    if STATE.get('pending'):
        raise RuntimeError('UNCERTAIN PENDING DELIVERY from a previous run: '+str(STATE['pending'])+'. Inspect Telegram before resuming; automatic retry may duplicate a post.')
    known={f"{unit['source_id']}:{unit['part']}" for unit in UNITS}
    extra=set(STATE['sent'])-known
    if extra: raise RuntimeError('Checkpoint has unexpected entries; refusing to send: '+str(list(extra)[:5]))
    def write_checkpoint():
        path=CHECKPOINT.with_suffix('.tmp')
        with path.open('w',encoding='utf8') as f:
            _json.dump(STATE,f,ensure_ascii=False,indent=2);f.flush();os.fsync(f.fileno())
        os.replace(path,CHECKPOINT)
    write_checkpoint()
    print('Confirmed units already sent:',len(STATE['sent']),'/',len(UNITS),flush=True)
    print('Sending to @southdhamal ONLY. Do not start another uploader at the same time.',flush=True)
    parent_by_source={}
    for unit in UNITS:
        sid,part=unit['source_id'],unit['part']
        key=f'{sid}:{part}'
        if key in STATE['sent']:
            if part==0: parent_by_source[sid]=STATE['sent'][key]['telegram_message_id']
            continue
        data={'chat_id':DEST['id']}
        kind=unit['kind']
        if part>0 and sid in parent_by_source:
            data['reply_to_message_id']=parent_by_source[sid]
            data['allow_sending_without_reply']='true'
        STATE['pending']={'source_id':sid,'part':part,'kind':kind,'time':time.time()}
        write_checkpoint()
        try:
            if kind=='text':
                data['text']=unit['text']
                data['disable_web_page_preview']='true'
                result=API.call('sendMessage',data,sending=True)
            else:
                data['caption']=unit['text']
                data['disable_notification']='true'
                photo_bytes=SOURCE_ZIP.read(ROOT+unit['media'])
                suffix=Path(unit['media']).suffix.lower()
                mime=mimetypes.guess_type(unit['media'])[0] or 'application/octet-stream'
                field='photo' if kind=='photo' else 'document'
                method='sendPhoto' if kind=='photo' else 'sendDocument'
                files={field:(Path(unit['media']).name,io.BytesIO(photo_bytes),mime)}
                result=API.call(method,data,files=files,sending=True)
        except Exception:
            print('Interrupted while attempting',key,'| review pending delivery before retry. Checkpoint:',CHECKPOINT,flush=True)
            raise
        if not isinstance(result,dict) or not result.get('message_id'):
            raise RuntimeError('Telegram returned no message ID, pending delivery requires manual review')
        STATE['sent'][key]={'telegram_message_id':result['message_id'],'kind':kind,'source_id':sid}
        STATE['pending']=None
        write_checkpoint()
        if part==0: parent_by_source[sid]=result['message_id']
        if len(STATE['sent'])%20==0 or len(STATE['sent'])==len(UNITS):
            print('SENT / checkpointed:',len(STATE['sent']),'/',len(UNITS),'| last original ID:',sid,flush=True)
        time.sleep(PAUSE_SECONDS)
    print('UPLOAD COMPLETE:',len(STATE['sent']),'/',len(UNITS), 'confirmed Telegram API messages',flush=True)
    print('Checkpoint:',CHECKPOINT)
    (WORK/'south_movie_upload_summary.json').write_text(json.dumps({**MANIFEST,'telegram_messages_confirmed':len(STATE['sent']),
                   'outbound_units':len(UNITS),'complete':len(STATE['sent'])==len(UNITS)},indent=2),encoding='utf-8')

In [ ]:
#@title 5 — Result / recovery instructions
print('South Movie work directory:',WORK)
print('Checkpoint (if sending started):',CHECKPOINT)
print('Source ZIP in temporary directory (not saved with notebook outputs):',ARCHIVE)
print('Source post count:',len(POSTS),'| outbound Telegram messages:',len(UNITS))
print('Missing original media / text-only fallbacks:', len(missing_media_audit),'/',text_fallback_count)
print('Media audit:', WORK/'south_movie_missing_media_audit.json')
print('This notebook never sends to the old Bollywood Movie channel ID:',OLD_BOLLYWOOD_CHANNEL_ID)
if CHECKPOINT.exists():
    state=json.loads(CHECKPOINT.read_text(encoding='utf-8'))
    print('Confirmed uploaded Telegram messages:',len(state['sent']), '/',len(UNITS))
    print('Pending uncertain send:',state.get('pending'))
print('For Kaggle: download Output / progress_south_movie_southdhamal.json after the version finishes.')
print('To resume an interrupted run, add that checkpoint file as an INPUT to the notebook before Save Version → Save & Run All.')
print('NEVER rerun from scratch after a partially completed upload; it can duplicate posts.')